# Early Stress Intervention Orchestrator## Notebook 3 of 3 — OrchestratorThe full Monday-morning autonomous loop. Reads moderate-and-severecustomers from Supabase, triages each through the agent stack, dispatchesto the chosen channel (EMAIL / CALL / ESCALATE), summarises responses,and produces a weekly executive memo.Run this once to seed 8 weeks of intervention history. Your Lovabledashboard will then show a fully populated system.**Prerequisites**: notebooks 01 and 02 already run.

In [ ]:
# Cell 1 — Setupfrom google.colab import drivedrive.mount('/content/drive')!pip install groq supabase sendgrid requests python-dotenv -q# Same keys as notebook 02SUPABASE_URL  = "your-supabase-url"SUPABASE_KEY  = "your-anon-key"GROQ_API_KEY  = "your-groq-key"SENDGRID_KEY  = "your-sendgrid-key"SENDGRID_FROM = "your-verified-sender@example.com"YOUR_EMAIL    = "your-personal-email@example.com"VAPI_KEY      = "your-vapi-key"VAPI_PHONE    = "your-vapi-phone-id"TEST_PHONE    = "+61XXXXXXXXX"RAILWAY_URL   = "your-railway-url"MODEL = "llama-3.3-70b-versatile"from groq import Groqfrom supabase import create_clientimport sendgridfrom sendgrid.helpers.mail import Mailimport requests, json, datetime, random, timegroq_client = Groq(api_key=GROQ_API_KEY)sb = create_client(SUPABASE_URL, SUPABASE_KEY)print("Setup complete")

In [ ]:
# Cell 2 — Paste all agent functions from notebook 02## This makes the orchestrator self-contained. Copy the function# definitions from 02_agents.ipynb Cells 2, 3, 4, 5, 6, 7:#   - triage_customer#   - send_email_intervention#   - place_call#   - generate_mock_transcript#   - summarise_interaction#   - generate_weekly_report## (Easiest: open both notebooks side by side and copy each function.)print("Once you've pasted the agent functions, run the next cell.")

In [ ]:
# Cell 3 — (Optional) Clear previous test runs## Uncomment to wipe interventions and weekly_reports before re-seeding.# sb.table('interventions').delete().neq('id', 0).execute()# sb.table('weekly_reports').delete().neq('id', 0).execute()# print("Cleared previous interventions and reports")

In [ ]:
# Cell 4 — Full 8-week seed run## Triages 80 customers (50 moderate + 30 severe), spread across 8 weeks.# Sends 3 real emails (to YOUR_EMAIL), drafts the rest without sending.# Uses mock transcripts for voice interventions (Vapi free-tier limit).moderate = sb.table('customers').select('*')\    .eq('risk_tier', 'Moderate').limit(50).execute().datasevere = sb.table('customers').select('*')\    .eq('risk_tier', 'Severe').limit(30).execute().dataall_customers = moderate + severerandom.shuffle(all_customers)print(f"Seeding interventions across 8 weeks for {len(all_customers)} customers...")print(f"  Moderate tier: {len(moderate)}  |  Severe tier: {len(severe)}\n")total_logged = 0weekly_logs = {w: [] for w in range(1, 9)}for idx, customer in enumerate(all_customers):    week = (idx % 8) + 1   # round-robin across 8 weeks    signals = sb.table('weekly_signals').select('*')\        .eq('customer_id', customer['customer_id'])\        .order('week', desc=True).limit(4).execute().data    try:        triage = triage_customer(customer, signals)    except Exception as e:        print(f"  [WARN] Triage failed for {customer['customer_id']}: {e}")        continue    action = triage['action']    print(f"  Wk{week} {customer['customer_id']} -> {action} ({triage['stress_assessment']})")    row = {        'customer_id': customer['customer_id'],        'week': week,        'action': action,        'triage_reasoning': triage['reasoning'],        'outcome': 'pending'    }    if action == 'MONITOR':        # Don't log MONITOR — these aren't interventions        continue    elif action == 'EMAIL':        try:            if total_logged < 3:                # Send 3 real emails                r = send_email_intervention(customer, triage)                row['email_subject'] = r['subject']                row['email_body'] = r['body']            else:                # Draft only — don't send (saves SendGrid quota)                draft = groq_client.chat.completions.create(                    model=MODEL,                    messages=[{"role": "user", "content": f"""Write a warm email from CommBank to {customer['name'].split()[0]}.Offer a free home loan health check call. Under 200 words.Return JSON: {{"subject": "...", "body_text": "..."}}"""}],                    response_format={"type": "json_object"},                    temperature=0.7                )                content = json.loads(draft.choices[0].message.content)                row['email_subject'] = content['subject']                row['email_body'] = content['body_text']            row['channel'] = 'email'            # Simulate that 60% of emails get a positive response            if random.random() < 0.6:                row['sentiment'] = random.choice(['positive', 'neutral'])                row['offer_accepted'] = random.choice(                    ['booked_call', 'none', 'hardship_referral'])                row['outcome'] = random.choice(['resolved', 'follow_up'])        except Exception as e:            print(f"    [WARN] Email failed: {e}")            continue    elif action == 'CALL':        try:            # Mock transcript (Vapi can't dial AU on free tier)            transcript = generate_mock_transcript(customer)            summary = summarise_interaction(transcript, customer, 'voice call')            row['channel'] = 'voice'            row['sentiment'] = summary['sentiment']            row['offer_accepted'] = summary['offer_accepted']            row['outcome'] = summary['outcome']            row['new_risk_signals'] = summary['new_risk_signals']            row['rm_briefing'] = summary.get('rm_briefing')        except Exception as e:            print(f"    [WARN] Call mock failed: {e}")            continue    elif action == 'ESCALATE':        row['channel'] = 'escalation'        row['outcome'] = 'escalate_to_human'        row['rm_briefing'] = triage.get('action_rationale',            f"Critical stress signals. Stress ratio: {customer['stress_ratio']:.2f}. "            "Recommended: hardship assessment, repayment plan review.")    sb.table('interventions').insert(row).execute()    weekly_logs[week].append(row)    total_logged += 1    time.sleep(0.3)  # Small delay to avoid rate limitsprint(f"\n[OK] {total_logged} interventions logged across 8 weeks.\n")

In [ ]:
# Cell 5 — Generate weekly reports for each weekprint("Generating weekly executive reports...")for week_num, week_log in weekly_logs.items():    if not week_log:        continue    try:        report = generate_weekly_report(week_num, week_log)        sb.table('weekly_reports').upsert({            'week': week_num,            'total_interventions': len(week_log),            'resolved': sum(1 for i in week_log if i.get('outcome') == 'resolved'),            'escalated': sum(1 for i in week_log if i.get('outcome') == 'escalate_to_human'),            'acceptance_rate': sum(1 for i in week_log                if i.get('offer_accepted', 'none') != 'none') / len(week_log),            'report_text': report        }).execute()        print(f"  Week {week_num}: {len(week_log)} interventions, report generated")    except Exception as e:        print(f"  [WARN] Week {week_num} report failed: {e}")print("\n[DONE] Database is fully seeded. Lovable dashboard will now show realistic data.")